Subject: ST 554 - Homework 10

Name: Franklin Zhou

Date: 4/18/2026

# Homework 10

For this homework you will create a gitHub repo (or use the one from previous mework) and save a python
notebook (`.ipynb` file) there. You must use our JupyterHub for this assignment as we’ll be using `pyspark`.
You’ll then submit a link to your gitHub repo.

You are tasked with using spark structured streaming to deal with streaming data!

## Part 1 - Creating Streaming Data Using `rate`

Setup a data stream using the `"rate"` format.

Prior to starting the stream, set up a sequence of actions using appropriate functions from `pyspark.sql.functions`
that uses the `rate` data to
- find the square root of the rate ‘value’
- find mod 4 of the rate ‘value’

To output this, create a `writeStream` that writes to ‘memory’ (`format("memory")`). Give the query a name
(`queryName("...")`) and start it!

Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the query
name (`spark.sql("select * from you_table_name").show()`).

In [7]:
# Load packages and initiate spark session
from pyspark.sql import SparkSession
import time
from pyspark.sql.functions import sqrt, col

spark = SparkSession.builder.getOrCreate()

In [8]:
# 1. Read a rate stream
rate_stream = spark.readStream.format("rate").load()

# 2. Transform: square root and mod 4 of 'value'
transformed = rate_stream \
    .withColumn("sqrt_value", sqrt(col("value"))) \
    .withColumn("mod4_value", col("value") % 4)

# 3. Write to memory
query1 = transformed.writeStream.format("memory").queryName("rate_table").outputMode("append").start()

# run 30 seconds
time.sleep(30)

# stop the query
query1.stop()

26/04/18 16:05:48 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e85e576f-af4e-4f73-bddb-68ef0b82283e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/18 16:05:48 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/04/18 16:06:18 WARN DAGScheduler: Failed to cancel job group fcbe98af-9105-4b99-aeb5-4d7c719e0a00. Cannot find active jobs for it.
26/04/18 16:06:18 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 30, writer: org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@21737eca] is aborting.
26/04/18 16:06:18 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 30, writer: org.apache.spark.sql.exe

In [10]:
# Output the table
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-18 16:05:...|    0|               0.0|         0|
|2026-04-18 16:05:...|    1|               1.0|         1|
|2026-04-18 16:05:...|    2|1.4142135623730951|         2|
|2026-04-18 16:05:...|    3|1.7320508075688772|         3|
|2026-04-18 16:05:...|    4|               2.0|         0|
|2026-04-18 16:05:...|    5|  2.23606797749979|         1|
|2026-04-18 16:05:...|    6| 2.449489742783178|         2|
|2026-04-18 16:05:...|    7|2.6457513110645907|         3|
|2026-04-18 16:05:...|    8|2.8284271247461903|         0|
|2026-04-18 16:05:...|    9|               3.0|         1|
|2026-04-18 16:05:...|   10|3.1622776601683795|         2|
|2026-04-18 16:05:...|   11|   3.3166247903554|         3|
|2026-04-18 16:06:...|   12|3.4641016151377544|         0|
|2026-04-18 16:06:...|   13| 3.605551275463989|         

## Part 2 - Using data from a CSV with a Pipeline

There are six `bikeDetails` sub datasets available on the assignment link. The one named `bikeDetails_for_fit.csv`
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following

- use an `SQLTransformer` with the following statement (this does some log transforms, renames a variable, and creates a dummy variable from categorical variable):

```
SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__
```

- use a `VectorAssembler` to create a `features` column. The `features` column should include the `year`, `log_km_driven`, and `one_owner` variables.
- create a `Pipeline` with the two steps above (`SQLTransformer` then `VectorAssembler`)
- fit this pipeline to the SQL data frame and save this as an object.

In [54]:
# Load packages
from pyspark.ml.feature import SQLTransformer, VectorAssembler
from pyspark.ml import Pipeline
import pandas as pd

# read data
bike_data = pd.read_csv("HW10/bikeDetails_for_fit.csv")
df = spark.createDataFrame(bike_data) # convert to spark sql data frame
df.show(5)


+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Royal Enfield Cla...|       175000|2019| Individual|1st owner|      350|              NaN|
|           Honda Dio|        45000|2017| Individual|1st owner|     5650|              NaN|
|Royal Enfield Cla...|       150000|2018| Individual|1st owner|    12000|         148114.0|
|Yamaha Fazer FI V...|        65000|2015| Individual|1st owner|    23000|          89643.0|
|Yamaha SZ [2013-2...|        20000|2011| Individual|2nd owner|    21000|              NaN|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
only showing top 5 rows


In [51]:
# SQL Transformer
sql_transformer = SQLTransformer(
    statement = """
        SELECT log(selling_price) AS label, year, log(km_driven) AS log_km_driven,
               CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
        FROM __THIS__
    """
)

# VectorAssembler: combine features
assembler = VectorAssembler(
    inputCols = ["year", "log_km_driven", "one_owner"],
    outputCol = "features"
)

# Build and fit pipeline
pipeline = Pipeline(stages = [sql_transformer, assembler])
fitted_pipeline = pipeline.fit(df)

Now we want to set up a read stream to look for csv files placed into a folder (the five `bikeDetails_add*.csv` files). When a csv comes in, we want to transform it using the fitted pipeline’s `.transform()` method! A few notes:

- You’ll need a schema to set up the `readStream`. You can use the SQL data frame’s schema from above! (`.schema` attribute)
- Each file you’ll be adding to the folder has a header!

We’ll just write the output to the ‘console’ using the ‘append’ mode.

Once that is all set up, make sure your folder you are looking for the .csv files is empty. Then start the
query. Place the other files into the folder one at a time. You should see the output appear below your
query. Once you’ve done all five files, stop the query!

This process will set us up well for using a fitted model to obtain predictions on streaming data!

In [25]:
df.schema

StructType([StructField('name', StringType(), True), StructField('selling_price', StringType(), True), StructField('year', StringType(), True), StructField('seller_type', StringType(), True), StructField('owner', StringType(), True), StructField('km_driven', StringType(), True), StructField('ex_showroom_price', StringType(), True)])

In [55]:
# get the schema
data_schema = df.schema

# read stream data
stream_df = spark.readStream.option("header", True).schema(data_schema).csv("HW10/csv_files")

# Apply the fitted pipeline to the incoming streaming data
stream_transformed = fitted_pipeline.transform(stream_df)

# Write output to console in append mode
query2 = stream_transformed \
    .writeStream \
    .format("console") \
    .outputMode("append") \
    .start()

26/04/18 17:19:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-67a4d161-4a3c-4ba0-ac6b-174fd42263f7. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/18 17:19:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
| 8.987196820661973|2003|10.887436932884098|        1|[2003.0,10.887436...|
|11.156250521031495|2018| 9.615805480084347|        1|[2018.0,9.6158054...|
|10.819778284410283|2016| 8.987196820661973|        1|[2016.0,8.9871968...|
| 10.46310334047155|2015|10.582738627903963|        1|[2015.0,10.582738...|
| 9.903487552536127|2006|11.225243392518447|        1|[2006.0,11.225243...|
|10.819778284410283|2012|10.239959789157341|        1|[2012.0,10.239959...|
| 10.51867319162636|2008| 11.03488966402723|        1|[2008.0,11.034889...|
|11.141861783579396|2018| 9.392661928770137|        1|[2018.0,9.3926619...|
|10.239959789157341|2012| 10.81975828421028|        1|[2012.0,10.81

In [56]:
query2.stop()

26/04/18 17:20:08 WARN DAGScheduler: Failed to cancel job group 8cb0c160-ba4c-46b2-81b1-5f06fcdae54c. Cannot find active jobs for it.
26/04/18 17:20:08 WARN DAGScheduler: Failed to cancel job group 8cb0c160-ba4c-46b2-81b1-5f06fcdae54c. Cannot find active jobs for it.
